Trying to explore what is possible in the open data scene!

In [2]:
import pandas as pd
from sodapy import Socrata
import re
import requests

In [3]:
# Dicts of ALL Chicago datasets with the API id used by client.get("xxxx-xxxx")

DOMAIN = "data.cityofchicago.org"
LIMIT = 1000
offset = 0

# name -> {"dataset_id": "xxxx-xxxx", "keys": [...]}
dataset_to_api = {}
dataset_columns = {}
while True:
    url = "https://api.us.socrata.com/api/catalog/v1"
    params = {
        "domains": DOMAIN,
        "only": "datasets",
        "limit": LIMIT,
        "offset": offset,
    }
    payload = requests.get(url, params=params, timeout=60).json()
    results = payload.get("results", [])
    if not results:
        break

    for item in results:
        resource = item.get("resource", {})
        name = resource.get("name")
        dataset_id = resource.get("id")  # use this in client.get(dataset_id, ... )
        keys = resource.get("columns_field_name") or []
        if name and dataset_id:
            dataset_to_api[name] = dataset_id
            dataset_columns[name] = keys

    offset += len(results)

print(f"Datasets found: {len(dataset_to_api):,}")

{"dataset_to_api": dataset_to_api, "dataset_columns": dataset_columns}

Datasets found: 898


{'dataset_to_api': {'Current Employee Names, Salaries, and Position Titles': 'xzkq-xp2w',
  'Building Permits': 'ydr8-5enu',
  'Crimes - 2001 to Present': 'ijzp-q8t2',
  'Traffic Crashes - Crashes': '85ca-t3if',
  'Food Inspections': '4ijn-s7e5',
  'Affordable Rental Housing Developments': 's6ha-ppgi',
  'Lobbyist Data - Historical - Lobbyist Registry - 2010': '2ft4-4uik',
  'Chicago Street Names': 'i6bp-fvbx',
  'Census Data - Selected socioeconomic indicators in Chicago, 2008 – 2012': 'kn9c-c2s2',
  'Business Licenses': 'r5kz-chrr',
  'Building Violations': '22u3-xenr',
  'Police Stations': 'z8bn-74gv',
  'Arrests': 'dpt3-jri9',
  'City-Owned Land Inventory': 'aksk-kvfp',
  'Chicago Public Schools - Progress Report Cards (2011-2012)': '9xs2-f89t',
  'Performance Metrics - Streets & Sanitation - Graffiti Removal': 'qcfn-tiw7',
  'Taxi Trips (2013-2023)': 'wrvz-psew',
  'Public Chauffeurs': '97wa-y6ff',
  'Towed Vehicles': 'ygr5-vcbg',
  'Chicago Police Department - Illinois Uniform Cr

In [4]:
# Metadata table: does each dataset have Date / Lat / Long columns? (0/1)


def _has_any(patterns, keys):
    for k in keys:
        key = str(k).lower()
        if any(re.search(p, key) for p in patterns):
            return 1
    return 0

def _matching_keys(patterns, keys):
    out = []
    for k in keys:
        key = str(k).lower()
        if any(re.search(p, key) for p in patterns):
            out.append(k)
    return out

date_patterns = [r"date", r"time", r"timestamp", r"datetime", r"reported"]
lat_patterns  = [r"^lat$", r"latitude", r"_lat", r"lat_"]
long_patterns = [r"^lon$", r"^long$", r"longitude", r"lng", r"_lon", r"_long"]

rows = []
for dataset_name, api_key in dataset_to_api.items():
    keys = dataset_columns.get(dataset_name, [])
    matched_date = _matching_keys(date_patterns, keys)
    matched_lat = _matching_keys(lat_patterns, keys)
    matched_long = _matching_keys(long_patterns, keys)

    rows.append({
        "dataset_name": dataset_name,
        "api_key": api_key,
        "Date": 1 if matched_date else 0,
        "Lat": 1 if matched_lat else 0,
        "Long": 1 if matched_long else 0,
        "n_columns": len(keys),
        "date_keys": matched_date,
        "lat_keys": matched_lat,
        "long_keys": matched_long,
    })

metadata_table = pd.DataFrame(rows).sort_values(["Date", "Lat", "Long", "n_columns"], ascending=[False, False, False, False]).reset_index(drop=True)

print("Datasets in metadata table:", len(metadata_table))
display(metadata_table.head(50))

# Optional focused view: only datasets with all three
datasets_with_date_lat_long = metadata_table[(metadata_table["Date"] == 1) & (metadata_table["Lat"] == 1) & (metadata_table["Long"] == 1)]
print("Datasets with Date + Lat + Long:", len(datasets_with_date_lat_long))
display(datasets_with_date_lat_long.head(20))

Datasets in metadata table: 898


,dataset_name,api_key,Date,Lat,Long,n_columns,date_keys,lat_keys,long_keys
0,Building Permits,ydr8-5enu,1,1,1,122,"[application_start_date, processing_time, issu...",[latitude],[longitude]
1,Chicago Public Schools - School Profile Inform...,9a5f-2r4p,1,1,1,104,"[open_for_enrollment_date, freshman_start_end_...",[school_latitude],[school_longitude]
2,Chicago Public Schools - School Profile Inform...,3dhs-m3w4,1,1,1,104,"[earliest_drop_off_time, closed_for_enrollment...",[school_latitude],[school_longitude]
3,Chicago Public Schools - School Profile Inform...,83yd-jxxw,1,1,1,101,"[earliest_drop_off_time, open_for_enrollment_d...",[school_latitude],[school_longitude]
4,Chicago Public Schools - School Profile Inform...,w4qj-h7bg,1,1,1,101,"[freshman_start_end_time, closed_for_enrollmen...",[school_latitude],[school_longitude]
5,Chicago Public Schools - School Profile Inform...,8i6r-et8s,1,1,1,100,"[freshman_start_end_time, earliest_drop_off_time]",[school_latitude],[school_longitude]
6,Chicago Public Schools - School Profile Inform...,cu4u-b4d9,1,1,1,99,"[open_for_enrollment_date, closed_for_enrollme...",[school_latitude],[school_longitude]
7,Chicago Public Schools - School Profile Inform...,2dem-8rq7,1,1,1,96,"[open_for_enrollment_date, closed_for_enrollme...",[school_latitude],[school_longitude]
8,Chicago Public Schools - School Profile Inform...,kh4r-387c,1,1,1,95,"[freshman_start_end_time, closed_for_enrollmen...",[school_latitude],[school_longitude]
9,Chicago Public Schools – High School Progress ...,qemx-3d9f,1,1,1,71,"[over_time_act_2012, over_time_explore_2010, o...",[latitude],[longitude]


Datasets with Date + Lat + Long: 92


,dataset_name,api_key,Date,Lat,Long,n_columns,date_keys,lat_keys,long_keys
0,Building Permits,ydr8-5enu,1,1,1,122,"[application_start_date, processing_time, issu...",[latitude],[longitude]
1,Chicago Public Schools - School Profile Inform...,9a5f-2r4p,1,1,1,104,"[open_for_enrollment_date, freshman_start_end_...",[school_latitude],[school_longitude]
2,Chicago Public Schools - School Profile Inform...,3dhs-m3w4,1,1,1,104,"[earliest_drop_off_time, closed_for_enrollment...",[school_latitude],[school_longitude]
3,Chicago Public Schools - School Profile Inform...,83yd-jxxw,1,1,1,101,"[earliest_drop_off_time, open_for_enrollment_d...",[school_latitude],[school_longitude]
4,Chicago Public Schools - School Profile Inform...,w4qj-h7bg,1,1,1,101,"[freshman_start_end_time, closed_for_enrollmen...",[school_latitude],[school_longitude]
5,Chicago Public Schools - School Profile Inform...,8i6r-et8s,1,1,1,100,"[freshman_start_end_time, earliest_drop_off_time]",[school_latitude],[school_longitude]
6,Chicago Public Schools - School Profile Inform...,cu4u-b4d9,1,1,1,99,"[open_for_enrollment_date, closed_for_enrollme...",[school_latitude],[school_longitude]
7,Chicago Public Schools - School Profile Inform...,2dem-8rq7,1,1,1,96,"[open_for_enrollment_date, closed_for_enrollme...",[school_latitude],[school_longitude]
8,Chicago Public Schools - School Profile Inform...,kh4r-387c,1,1,1,95,"[freshman_start_end_time, closed_for_enrollmen...",[school_latitude],[school_longitude]
9,Chicago Public Schools – High School Progress ...,qemx-3d9f,1,1,1,71,"[over_time_act_2012, over_time_explore_2010, o...",[latitude],[longitude]


## Schools

In [13]:
if "School" in datasets_with_date_lat_long['dataset_name'].str.cat(sep=' '):
    print("Dataset about schools with Date + Lat + Long:")
    
schools_datasets = datasets_with_date_lat_long[datasets_with_date_lat_long['dataset_name'].str.contains("Schools", case=False, na=False)]

school_api_keys = schools_datasets['api_key'].tolist()
school_api_keys = {schools_datasets.iloc[i]['dataset_name']: schools_datasets.iloc[i]['api_key'] for i in range(len(schools_datasets))}
schools_data = {}

schools_data = {}
client = Socrata(DOMAIN, None)  # No app token, read-only access
for dataset_name, api_key in school_api_keys.items():
    print(f"Fetching data for dataset: {dataset_name} ({api_key})")
    
    try:
        data = client.get(api_key, limit=2000)
        print(f"Sample data from dataset {dataset_name}:")
        schools_data[dataset_name] = data
        
    except Exception as e:
        print(f"Error fetching data for dataset {dataset_name}: {e}")



Dataset about schools with Date + Lat + Long:
Fetching data for dataset: Chicago Public Schools - School Profile Information SY2223 (9a5f-2r4p)
Sample data from dataset Chicago Public Schools - School Profile Information SY2223:
Fetching data for dataset: Chicago Public Schools - School Profile Information SY2425 (3dhs-m3w4)
Sample data from dataset Chicago Public Schools - School Profile Information SY2425:
Fetching data for dataset: Chicago Public Schools - School Profile Information SY2021 (83yd-jxxw)
Sample data from dataset Chicago Public Schools - School Profile Information SY2021:
Fetching data for dataset: Chicago Public Schools - School Profile Information SY1718 (w4qj-h7bg)
Sample data from dataset Chicago Public Schools - School Profile Information SY1718:
Fetching data for dataset: Chicago Public Schools - School Profile Information SY1617 (8i6r-et8s)
Sample data from dataset Chicago Public Schools - School Profile Information SY1617:
Fetching data for dataset: Chicago Publ

In [21]:
schools_data.values()

dict_values([[{'school_id': '400147', 'legacy_unit_id': '9066', 'finance_id': '63141', 'short_name': 'CHICAGO EXCEL HS', 'long_name': 'Chicago Excel Academy HS', 'primary_category': 'HS', 'is_high_school': True, 'is_middle_school': False, 'is_elementary_school': False, 'is_pre_school': False, 'summary': 'School', 'administrator_title': 'Principal', 'administrator': 'Tyree Booker', 'address': '1257 W 111TH ST', 'city': 'Chicago', 'state': 'Illinois', 'zip': '60643', 'phone': '773-629-8379', 'fax': '773-629-8736', 'cps_school_profile': {'url': 'https://www.cps.edu/schools/schoolprofiles/400147'}, 'website': {'url': 'https://cameloteducation.org/excelroseland'}, 'attendance_boundaries': False, 'grades_offered_all': '9,10,11,12', 'grades_offered': '9-12', 'student_count_total': '208', 'student_count_low_income': '155', 'student_count_special_ed': '20', 'student_count_english_learners': '1', 'student_count_black': '199', 'student_count_hispanic': '5', 'student_count_white': '0', 'student_co

In [14]:
for dataset_name, data in schools_data.items():
    print(f"\nDataset: {dataset_name}")
    if data:
        print(f"Number of records: {len(data)}")
        print(f"Number of columns: {len(data[0]) if len(data) > 0 else 0}")
        
    else:
        print("No data available for this dataset.")


Dataset: Chicago Public Schools - School Profile Information SY2223
Number of records: 655
Number of columns: 66

Dataset: Chicago Public Schools - School Profile Information SY2425
Number of records: 652
Number of columns: 69

Dataset: Chicago Public Schools - School Profile Information SY2021
Number of records: 655
Number of columns: 68

Dataset: Chicago Public Schools - School Profile Information SY1718
Number of records: 661
Number of columns: 62

Dataset: Chicago Public Schools - School Profile Information SY1617
Number of records: 661
Number of columns: 66

Dataset: Chicago Public Schools - School Profile Information SY2324
Number of records: 652
Number of columns: 67

Dataset: Chicago Public Schools - School Profile Information SY2122
Number of records: 655
Number of columns: 65

Dataset: Chicago Public Schools - School Profile Information SY1819
Number of records: 654
Number of columns: 71

Dataset: Chicago Public Schools ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“ High School Progress Report Card (20

In [15]:
# For each dataset, plot the records over spce (lat/long) 
# Should make 1 map with 9 toggles for each dataset

extract_latlong_columns = {}
for dataset_name, data in schools_data.items():
    keys = dataset_columns.get(dataset_name, [])
    matched_lat = _matching_keys(lat_patterns, keys)
    matched_long = _matching_keys(long_patterns, keys)
    extract_latlong_columns[dataset_name] = {"lat": matched_lat, "long": matched_long} 
print(extract_latlong_columns)

{'Chicago Public Schools - School Profile Information SY2223': {'lat': ['school_latitude'], 'long': ['school_longitude']}, 'Chicago Public Schools - School Profile Information SY2425': {'lat': ['school_latitude'], 'long': ['school_longitude']}, 'Chicago Public Schools - School Profile Information SY2021': {'lat': ['school_latitude'], 'long': ['school_longitude']}, 'Chicago Public Schools - School Profile Information SY1718': {'lat': ['school_latitude'], 'long': ['school_longitude']}, 'Chicago Public Schools - School Profile Information SY1617': {'lat': ['school_latitude'], 'long': ['school_longitude']}, 'Chicago Public Schools - School Profile Information SY2324': {'lat': ['school_latitude'], 'long': ['school_longitude']}, 'Chicago Public Schools - School Profile Information SY2122': {'lat': ['school_latitude'], 'long': ['school_longitude']}, 'Chicago Public Schools - School Profile Information SY1819': {'lat': ['school_latitude'], 'long': ['school_longitude']}, 'Chicago Public Schools

In [22]:
# Inspect what data is available per school dataset
import pandas as pd
import re

def summarize_school_dataset(dataset_name, data):
    print(f"\n=== {dataset_name} ===")
    if not data:
        print("No rows.")
        return

    df = pd.DataFrame(data)
    print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")

    # Show likely useful columns first
    interesting_patterns = [
        r"school|name", r"zip|postal", r"address|street", r"phone",
        r"website|web", r"grade", r"enroll", r"lat|lon|long|lng|geo"
    ]
    score = {}
    for c in df.columns:
        lc = c.lower()
        score[c] = sum(bool(re.search(p, lc)) for p in interesting_patterns)

    ranked = sorted(df.columns, key=lambda c: (-score[c], c))
    print("Top candidate columns:")
    print(ranked[:20])

    # Non-null coverage
    coverage = (df.notna().mean() * 100).sort_values(ascending=False)
    print("\nNon-null coverage (top 15):")
    print(coverage.head(15).round(1).to_string())

    print("\nSample rows:")
    display(df[ranked[:10]].head(3))

for ds_name, ds_data in schools_data.items():
    summarize_school_dataset(ds_name, ds_data)


=== Chicago Public Schools - School Profile Information SY2223 ===
Rows: 655 | Columns: 97
Top candidate columns:
['college_enrollment_rate_school', 'long_name', 'school_latitude', 'school_longitude', 'address', 'after_school_hours', 'college_enrollment_rate_mean', 'cps_school_profile', 'fifth_contact_name', 'fourth_contact_name', 'grades_offered', 'grades_offered_all', 'graduation_rate_school', 'is_elementary_school', 'is_gocps_high_school', 'is_high_school', 'is_middle_school', 'is_pre_school', 'kindergarten_school_day', 'phone']

Non-null coverage (top 15):
school_id               100.0
legacy_unit_id          100.0
finance_id              100.0
short_name              100.0
long_name               100.0
primary_category        100.0
is_high_school          100.0
is_middle_school        100.0
is_elementary_school    100.0
is_pre_school           100.0
administrator_title     100.0
city                    100.0
address                 100.0
state                   100.0
zip         

,college_enrollment_rate_school,long_name,school_latitude,school_longitude,address,after_school_hours,college_enrollment_rate_mean,cps_school_profile,fifth_contact_name,fourth_contact_name
0,31.3,Chicago Excel Academy HS,41.691953,-87.654248,1257 W 111TH ST,NaN,67.2,{'url': 'https://www.cps.edu/schools/schoolpro...,NaN,NaN
1,NaN,Ole A Thorp Elementary Scholastic Academy,41.948571,-87.777631,6024 W WARWICK AVE,Paid After-care 6:30 PM,NaN,{'url': 'https://www.cps.edu/schools/schoolpro...,NaN,NaN
2,66.5,Sarah E. Goode STEM Academy,41.75375,-87.707345,7651 S HOMAN AVE,3:00pm - 5:00pm,67.2,{'url': 'https://www.cps.edu/schools/schoolpro...,NaN,NaN



=== Chicago Public Schools - School Profile Information SY2425 ===
Rows: 652 | Columns: 102
Top candidate columns:
['college_enrollment_rate_school', 'long_name', 'school_latitude', 'school_longitude', 'address', 'after_school_hours', 'closed_for_enrollment_date', 'college_enrollment_rate_mean', 'cps_school_profile', 'fifth_contact_name', 'fourth_contact_name', 'grades_offered', 'grades_offered_all', 'graduation_rate_school', 'is_elementary_school', 'is_gocps_high_school', 'is_high_school', 'is_middle_school', 'is_pre_school', 'kindergarten_school_day']

Non-null coverage (top 15):
school_id                 100.0
legacy_unit_id            100.0
finance_id                100.0
short_name                100.0
long_name                 100.0
primary_category          100.0
is_high_school            100.0
is_middle_school          100.0
is_elementary_school      100.0
is_pre_school             100.0
administrator_title       100.0
address                   100.0
cps_school_profile        

,college_enrollment_rate_school,long_name,school_latitude,school_longitude,address,after_school_hours,closed_for_enrollment_date,college_enrollment_rate_mean,cps_school_profile,fifth_contact_name
0,NaN,Alain Locke Charter School,41.877248,-87.705235,3141 W JACKSON BLVD,NaN,NaN,NaN,{'url': 'https://www.cps.edu/schools/schoolpro...,NaN
1,NaN,Acero Charter Schools - Octavio Paz Campus,41.850035,-87.692551,2651 W 23RD ST,NaN,NaN,NaN,{'url': 'https://www.cps.edu/schools/schoolpro...,NaN
2,NaN,Acero Charter Schools - Rufino Tamayo,41.79993,-87.693862,5135 S CALIFORNIA AVE,NaN,NaN,NaN,{'url': 'https://www.cps.edu/schools/schoolpro...,NaN



=== Chicago Public Schools - School Profile Information SY2021 ===
Rows: 655 | Columns: 99
Top candidate columns:
['college_enrollment_rate_school', 'long_name', 'school_latitude', 'school_longitude', 'address', 'after_school_hours', 'closed_for_enrollment_date', 'college_enrollment_rate_mean', 'cps_school_profile', 'fifth_contact_name', 'fourth_contact_name', 'grades_offered', 'grades_offered_all', 'graduation_rate_school', 'is_elementary_school', 'is_gocps_high_school', 'is_high_school', 'is_middle_school', 'is_pre_school', 'kindergarten_school_day']

Non-null coverage (top 15):
school_id               100.0
legacy_unit_id          100.0
finance_id              100.0
short_name              100.0
long_name               100.0
primary_category        100.0
is_high_school          100.0
is_middle_school        100.0
is_elementary_school    100.0
is_pre_school           100.0
administrator_title     100.0
zip                     100.0
address                 100.0
city                 

,college_enrollment_rate_school,long_name,school_latitude,school_longitude,address,after_school_hours,closed_for_enrollment_date,college_enrollment_rate_mean,cps_school_profile,fifth_contact_name
0,NaN,Providence Englewood Charter School,41.775367,-87.663996,6515 S ASHLAND AVE,NaN,NaN,NaN,{'url': 'https://www.cps.edu/schools/schoolpro...,NaN
1,NaN,Joseph Brennemann Elementary School,41.959984,-87.649749,4251 N CLARENDON AVE,NaN,NaN,NaN,{'url': 'https://www.cps.edu/schools/schoolpro...,NaN
2,NaN,Joshua D Kershaw Elementary School,41.776453,-87.640892,6450 S LOWE AVE,NaN,NaN,NaN,{'url': 'https://www.cps.edu/schools/schoolpro...,NaN


... (8 outputs left out)


=== Chicago Public Schools - School Profile Information SY1819 ===
Rows: 654 | Columns: 92
Top candidate columns:
['college_enrollment_rate_school', 'long_name', 'school_latitude', 'school_longitude', 'address', 'after_school_hours', 'college_enrollment_rate_mean', 'cps_school_profile', 'fifth_contact_name', 'fourth_contact_name', 'grades_offered', 'grades_offered_all', 'graduation_rate_school', 'is_elementary_school', 'is_gocps_high_school', 'is_high_school', 'is_middle_school', 'is_pre_school', 'kindergarten_school_day', 'open_for_enrollment_date']

Non-null coverage (top 15):
school_id               100.0
legacy_unit_id          100.0
finance_id              100.0
short_name              100.0
long_name               100.0
primary_category        100.0
is_high_school          100.0
is_middle_school        100.0
is_elementary_school    100.0
is_pre_school           100.0
administrator_title     100.0
address                 100.0
student_count_total     100.0
zip                    

,college_enrollment_rate_school,long_name,school_latitude,school_longitude,address,after_school_hours,college_enrollment_rate_mean,cps_school_profile,fifth_contact_name,fourth_contact_name
0,NaN,Charles G Hammond Elementary School,41.852719,-87.696243,2819 W 21ST PL,2:30 pm - 5:30 pm,NaN,https://www.cps.edu/schools/schoolprofiles/609966,NaN,NaN
1,NaN,Polaris Charter Academy,41.89255,-87.707796,620 N SAWYER AVE,4:00 pm-6:00 pm,NaN,https://www.cps.edu/schools/schoolprofiles/400069,NaN,NaN
2,NaN,Stone Elementary Scholastic Academy,41.995359,-87.684793,6239 N LEAVITT ST,NaN,NaN,https://www.cps.edu/schools/schoolprofiles/610191,NaN,NaN



=== Chicago Public Schools ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“ High School Progress Report Card (2012-2013) ===
Rows: 92 | Columns: 61
Top candidate columns:
['college_enrollment_2011_percent', 'college_enrollment_2012_percent', 'geographic_network', 'growth_all_grades', 'healthy_school_certified', 'latitude', 'longitude', 'school_id', 'school_name', 'school_short_name', 'school_track', 'street_address', 'website', 'zip', '_5_year_cohort_graduation_rate_2011_percent', '_5_year_cohort_graduation_rate_2012_percent', 'ambitious_instruction', 'average_days_of_suspension_2011', 'average_days_of_suspension_2012', 'average_score_act_2012']

Non-null coverage (top 15):
school_id                    100.0
school_short_name            100.0
school_name                  100.0
street_address               100.0
city                         100.0
state                        100.0
zip                          100.0
managed_by_network           100.0
geographic_network           100.0
overall_rating               10

,college_enrollment_2011_percent,college_enrollment_2012_percent,geographic_network,growth_all_grades,healthy_school_certified,latitude,longitude,school_id,school_name,school_short_name
0,64.20,75.80,North-Northwest Side High School Network,NaN,No,41.94215439,-87.77650575,610340,Chicago Academy High School,Chgo Acad HS
1,51.30,54.10,West Side High School Network,64.00,No,41.88520477,-87.76321191,400018,Austin Business and Entrepreneurship Academy HS,Austin Bus & Entrp HS
2,NaN,NaN,North-Northwest Side High School Network,61.10,No,41.93576106,-87.68052441,610524,Alcott High School for the Humanities,Alcott HS


In [ ]:
import folium
from folium.plugins import MarkerCluster
import re

def first_present(record, keys):
    for k in keys:
        v = record.get(k)
        if v is not None and str(v).strip() != "":
            return v
    return None

def to_float(v):
    try:
        return float(v)
    except Exception:
        return None

def normalize_zip(v):
    if v is None:
        return "Unknown ZIP"
    m = re.search(r"\b(\d{5})\b", str(v))
    return m.group(1) if m else "Unknown ZIP"

name_patterns = [r"school_name", r"school", r"facility_name", r"site_name", r"^name$"]
zip_patterns = [r"zip", r"zipcode", r"postal"]
popup_patterns = [r"address", r"city", r"state", r"phone", r"website", r"grade", r"enroll"]

chicago_map = folium.Map(location=[41.8781, -87.6298], zoom_start=11)

for dataset_name, data in schools_data.items():
    if not data:
        continue

    keys = dataset_columns.get(dataset_name, [])
    lat_keys = _matching_keys(lat_patterns, keys)
    long_keys = _matching_keys(long_patterns, keys)
    name_keys = _matching_keys(name_patterns, keys)
    zip_keys = _matching_keys(zip_patterns, keys)
    popup_keys = _matching_keys(popup_patterns, keys)

    if not lat_keys or not long_keys:
        print(f"Skipping {dataset_name}: missing lat/long columns.")
        continue

    dataset_group = folium.FeatureGroup(name=dataset_name, show=False).add_to(chicago_map)
    zip_clusters = {}  # zip -> MarkerCluster

    for record in data:
        lat = to_float(first_present(record, lat_keys))
        lon = to_float(first_present(record, long_keys))
        if lat is None or lon is None:
            continue

        school_name = first_present(record, name_keys) or "Unknown School"
        zip_code = normalize_zip(first_present(record, zip_keys))

        if zip_code not in zip_clusters:
            zip_clusters[zip_code] = MarkerCluster(name=f"{dataset_name} | ZIP {zip_code}").add_to(dataset_group)

        lines = [f"<b>{school_name}</b>", f"ZIP: {zip_code}"]
        for k in popup_keys[:6]:
            v = record.get(k)
            if v is not None and str(v).strip() != "":
                lines.append(f"{k}: {v}")

        popup_html = "<br>".join(lines)
        folium.Marker(
            location=[lat, lon],
            popup=folium.Popup(popup_html, max_width=350),
            tooltip=school_name
        ).add_to(zip_clusters[zip_code])

folium.LayerControl(collapsed=False).add_to(chicago_map)
chicago_map.save("chicago_schools_map.html")
print("Map saved: chicago_schools_map.html")

Map has been saved to chicago_schools_map.html


In [6]:
# Energy datasets: fetch all year-specific benchmarking + covered buildings (in-memory only)
energy_dataset_ids = {
    "Chicago Energy Benchmarking - 2014 Data Reported in 2015": "tepd-j7h5",
    "Chicago Energy Benchmarking - 2015 Data Reported in 2016": "ebtp-548e",
    "Chicago Energy Benchmarking - 2016 Data Reported in 2017": "fpwt-snya",
    "Chicago Energy Benchmarking - 2017 Data Reported in 2018": "j2ev-2azp",
    "Chicago Energy Benchmarking - 2018 Data Reported in 2019": "m2kv-bmi3",
    "Chicago Energy Benchmarking - 2019 Data Reported in 2020": "jn94-it7m",
    "Chicago Energy Benchmarking - 2020 Data Reported in 2021": "ydbk-8hi6",
    "Chicago Energy Benchmarking - 2021 Data Reported in 2022": "gkf4-txtp",
    "Chicago Energy Benchmarking - 2022 Data Reported in 2023": "mz3g-jagv",
    "Chicago Energy Benchmarking - 2023 Data Reported in 2024": "3a36-5x9a",
    "Chicago Energy Benchmarking - Covered Buildings": "g5i5-yz37",
}

client = Socrata(DOMAIN, None)

def fetch_all_socrata_rows(api_key, page_size=50000):
    all_rows = []
    offset = 0
    while True:
        batch = client.get(api_key, limit=page_size, offset=offset)
        if not batch:
            break
        all_rows.extend(batch)
        if len(batch) < page_size:
            break
        offset += page_size
    return all_rows

energy_data = {}
for dataset_name, api_key in energy_dataset_ids.items():
    print(f"Fetching: {dataset_name} ({api_key})")
    try:
        records = fetch_all_socrata_rows(api_key)
        energy_data[dataset_name] = pd.DataFrame.from_records(records)
        print(f"columns: {energy_data[dataset_name].columns.tolist()} | ", end="")
        print(f"Rows: {len(energy_data[dataset_name]):,}")
    except Exception as e:
        print(f"Error for {dataset_name}: {e}")
        energy_data[dataset_name] = pd.DataFrame()

energy_summary = pd.DataFrame([
    {
        "dataset_name": name,
        "api_key": energy_dataset_ids[name],
        "rows": len(df),
        "columns": len(df.columns),
    }
    for name, df in energy_data.items()
]).sort_values("dataset_name").reset_index(drop=True)

print("\nEnergy dataset fetch complete.")
display(energy_summary)

benchmark_2023 = energy_data["Chicago Energy Benchmarking - 2023 Data Reported in 2024"]
covered_buildings = energy_data["Chicago Energy Benchmarking - Covered Buildings"]

print("\nPreview: 2023 benchmarking")
display(benchmark_2023.head(5))

print("\nPreview: covered buildings")
display(covered_buildings.head(5))

Fetching: Chicago Energy Benchmarking - 2014 Data Reported in 2015 (tepd-j7h5)
columns: ['data_year', 'id', 'property_name', 'reporting_status', 'address', 'zip_code', 'community_area', 'primary_property_type', 'gross_floor_area_buildings_sq_ft', 'year_built', 'of_buildings', 'electricity_use_kbtu', 'natural_gas_use_kbtu', 'site_eui_kbtu_sq_ft', 'source_eui_kbtu_sq_ft', 'total_ghg_emissions_metric_tons_co2e', 'ghg_intensity_kg_co2e_sq_ft', 'latitude', 'longitude', 'location', 'row_id', 'energy_star_score', 'weather_normalized_site_eui_kbtu_sq_ft', 'weather_normalized_source_eui_kbtu_sq_ft', 'district_steam_use_kbtu', 'district_chilled_water_use_kbtu', 'all_other_fuel_use_kbtu'] | Rows: 243
Fetching: Chicago Energy Benchmarking - 2015 Data Reported in 2016 (ebtp-548e)
columns: ['data_year', 'benchmarking_id', 'property_name', 'address', 'zip_code', 'community_area', 'primary_property_type_epa_calculated', 'property_gross_floor_area_epa_calculated_buildings_sq_ft', 'year_built', 'number_

,dataset_name,api_key,rows,columns
0,Chicago Energy Benchmarking - 2014 Data Report...,tepd-j7h5,243,27
1,Chicago Energy Benchmarking - 2015 Data Report...,ebtp-548e,1521,25
2,Chicago Energy Benchmarking - 2016 Data Report...,fpwt-snya,2717,25
3,Chicago Energy Benchmarking - 2017 Data Report...,j2ev-2azp,2797,25
4,Chicago Energy Benchmarking - 2018 Data Report...,m2kv-bmi3,3430,29
5,Chicago Energy Benchmarking - 2019 Data Report...,jn94-it7m,3438,27
6,Chicago Energy Benchmarking - 2020 Data Report...,ydbk-8hi6,3582,27
7,Chicago Energy Benchmarking - 2021 Data Report...,gkf4-txtp,3550,33
8,Chicago Energy Benchmarking - 2022 Data Report...,mz3g-jagv,3613,28
9,Chicago Energy Benchmarking - 2023 Data Report...,3a36-5x9a,3438,29



Preview: 2023 benchmarking


,data_year,id,property_name,reporting_status,address,zip_code,chicago_energy_rating,community_area,primary_property_type,gross_floor_area_buildings_sq_ft,...,total_ghg_emissions_metric_tons_co2e,ghg_intensity_kg_co2e_sq_ft,latitude,longitude,location,row_id,district_steam_use_kbtu,exempt_from_chicago_energy_rating,water_use_kgal,district_chilled_water_use_kbtu
0,2023,101710,AlgonOuin #1,Submitted,1605 E. 50th Street,60615,1,Kenwood,Multifamily Housing,59310,...,532.6,9,41.910944,-87.626393,"{'type': 'Point', 'coordinates': [-87.626393, ...",2023-101710,NaN,NaN,NaN,NaN
1,2023,160211,633 N St. Clair,Submitted,633 N St. Clair,60611,4,Near North Side,Office,632163,...,3079.7,4.9,41.747150,-87.628002,"{'type': 'Point', 'coordinates': [-87.628002, ...",2023-160211,NaN,NaN,NaN,NaN
2,2023,239016,Avenue East Condominiums,Submitted,160 East Illinois Street,60611,2.5,Near North Side,Multifamily Housing,258647,...,1406.4,5.4,41.790900,-87.671932,"{'type': 'Point', 'coordinates': [-87.671932, ...",2023-239016,NaN,NaN,NaN,NaN
3,2023,252049,Bond Chapel,Submitted,1025 East 58th Street,60637,4,Hyde Park,Worship Facility,5252,...,62,11.8,41.924508,-87.638223,"{'type': 'Point', 'coordinates': [-87.638223, ...",2023-252049,707421.1,NaN,NaN,NaN
4,2023,250120,Sinai Schwab Rehabilitation Hospital,Submitted,1401 S California Ave,60608,2,North Lawndale,Outpatient Rehabilitation/Physical Therapy,177380,...,1629.1,9.2,41.927891,-87.736002,"{'type': 'Point', 'coordinates': [-87.736002, ...",2023-250120,NaN,NaN,NaN,NaN



Preview: covered buildings


,building_id,address,zip,cohort_sector,cohort_size,verification_year,community_area_name,community_area_number,ward,latitude,longitude,location,:@computed_region_vrxf_vc4k,:@computed_region_6mkv_f3dw,:@computed_region_rpca_8um6,:@computed_region_bdys_3d7i,:@computed_region_43wa_7qmu,:@computed_region_awaf_s7ux
0,251626,4929 N Sawyer Ave,60625,Municipal,"> 50,000 Sq Ft",2025,Albany Park,14,33,41.97110,-87.70939,"{'latitude': '41.9711', 'longitude': '-87.70939'}",14,21849,20,750,20,28
1,250014,2949 E 131st St,60633,Municipal,"> 50,000 Sq Ft",2025,Hegewisch,55,10,41.65732,-87.55272,"{'latitude': '41.65732', 'longitude': '-87.552...",52,22250,26,586,47,47
2,105044,2018 2040 W Farwell Ave,60645,Residential,"> 50,000 Sq Ft",2025,West Ridge,2,50,42.00880,-87.69503,"{'latitude': '42.0088', 'longitude': '-87.69503'}",20,22528,12,42,27,42
3,254316,2544 W Montrose Ave,60618,Commercial,"> 50,000 Sq Ft",2027,Lincoln Square,4,47,41.94662,-87.70269,"{'latitude': '41.94662', 'longitude': '-87.702...",22,21538,39,425,20,2
4,251445,4619 S Wolcott Ave,60609,Municipal,"> 50,000 Sq Ft",2025,New City,61,15,41.80974,-87.67190,"{'latitude': '41.80974', 'longitude': '-87.6719'}",59,14924,37,706,3,53


In [7]:
# Interactive energy map explorer (Cell 13)
import numpy as np
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

requested_fields = [
    "community_area",
    "primary_property_type_epa_calculated",
    "property_gross_floor_area_epa_calculated_buildings_sq_ft",
    "year_built",
    "number_of_buildings",
    "electricity_use_grid_purchase_and_generated_from_onsite_renewable_systems_kbtu",
    "natural_gas_use_kbtu",
    "site_eui_kbtu_sq_ft",
    "source_eui_kbtu_sq_ft",
    "weather_normalized_site_eui_kbtu_sq_ft",
    "weather_normalized_source_eui_kbtu_sq_ft",
    "total_ghg_emissions_metric_tons_co2e",
    "ghg_intensity_metric_tons_co2e_sq_ft",
    "latitude",
    "longitude",
    "location",
    "energy_star_score",
    "district_chilled_water_use_kbtu",
    "district_steam_use_kbtu",
    "all_other_fuel_use_kbtu",
]

numeric_fields = [
    "property_gross_floor_area_epa_calculated_buildings_sq_ft",
    "year_built",
    "number_of_buildings",
    "electricity_use_grid_purchase_and_generated_from_onsite_renewable_systems_kbtu",
    "natural_gas_use_kbtu",
    "site_eui_kbtu_sq_ft",
    "source_eui_kbtu_sq_ft",
    "weather_normalized_site_eui_kbtu_sq_ft",
    "weather_normalized_source_eui_kbtu_sq_ft",
    "total_ghg_emissions_metric_tons_co2e",
    "ghg_intensity_metric_tons_co2e_sq_ft",
    "energy_star_score",
    "district_chilled_water_use_kbtu",
    "district_steam_use_kbtu",
    "all_other_fuel_use_kbtu",
]

def normalize_energy_df(df):
    work = df.copy()

    if "location" in work.columns and ("latitude" not in work.columns or "longitude" not in work.columns):
        lat_vals = []
        lon_vals = []
        for val in work["location"].tolist():
            if isinstance(val, dict):
                lat_vals.append(val.get("latitude"))
                lon_vals.append(val.get("longitude"))
            else:
                lat_vals.append(np.nan)
                lon_vals.append(np.nan)
        if "latitude" not in work.columns:
            work["latitude"] = lat_vals
        if "longitude" not in work.columns:
            work["longitude"] = lon_vals

    for c in numeric_fields + ["latitude", "longitude"]:
        if c in work.columns:
            work[c] = pd.to_numeric(work[c], errors="coerce")

    if "latitude" in work.columns and "longitude" in work.columns:
        work = work.dropna(subset=["latitude", "longitude"])
        work = work[(work["latitude"].between(41.55, 42.10)) & (work["longitude"].between(-88.0, -87.2))]

    return work

energy_plot_data = {name: normalize_energy_df(df) for name, df in energy_data.items()}

# all_community_values = sorted({str(v) for d in energy_plot_data.values() if "community_area" in d.columns for v in d["community_area"].dropna().unique()})
all_property_values = sorted({str(v) for d in energy_plot_data.values() if "primary_property_type_epa_calculated" in d.columns for v in d["primary_property_type_epa_calculated"].dropna().unique()})

dataset_dropdown = widgets.Dropdown(options=list(energy_plot_data.keys()), value=list(energy_plot_data.keys())[0], description="Dataset:", layout=widgets.Layout(width="70%"))
color_dropdown = widgets.Dropdown(options=["site_eui_kbtu_sq_ft"] + [c for c in numeric_fields if c != "site_eui_kbtu_sq_ft"], value="site_eui_kbtu_sq_ft", description="Color:", layout=widgets.Layout(width="70%"))
# size_dropdown = widgets.Dropdown(options=["None"] + numeric_fields, value="None", description="Size:", layout=widgets.Layout(width="70%"))
# community_multi = widgets.SelectMultiple(options=all_community_values, value=(), description="Community:", rows=8, layout=widgets.Layout(width="47%"))
property_multi = widgets.SelectMultiple(options=all_property_values, value=(), description="Property type:", rows=8, layout=widgets.Layout(width="47%"))
# max_points_slider = widgets.IntSlider(value=6000, min=500, max=12000, step=500, description="Max points:", continuous_update=False, layout=widgets.Layout(width="70%"))
status_html = widgets.HTML()

def make_interactive_map(dataset_name, color_col, selected_property):
    df = energy_plot_data.get(dataset_name, pd.DataFrame()).copy()

    if df.empty:
        status_html.value = "<b>No rows available after coordinate normalization.</b>"
        return px.scatter_mapbox(pd.DataFrame({"latitude": [], "longitude": []}), lat="latitude", lon="longitude", zoom=10)

    # if selected_community and "community_area" in df.columns:
    #     selected = set(selected_community)
    #     df = df[df["community_area"].astype(str).isin(selected)]

    # if selected_property and "primary_property_type_epa_calculated" in df.columns:
    #     selected = set(selected_property)
    #     df = df[df["primary_property_type_epa_calculated"].astype(str).isin(selected)]

    if df.empty:
        status_html.value = "<b>No rows match current filters.</b>"
        return px.scatter_mapbox(pd.DataFrame({"latitude": [], "longitude": []}), lat="latitude", lon="longitude", zoom=10)


    hover_cols = [c for c in requested_fields if c in df.columns and c != "location"]
    use_color = color_col if color_col in df.columns else None
    # use_size = size_col if size_col != "None" and size_col in df.columns else None

    title = f"{dataset_name}<br><sup>{len(df):,} points shown | color={use_color or 'none'} </sup>"
    fig = px.scatter_map(
        df,
        lat="latitude",
        lon="longitude",
        color=use_color,
        # size=use_size,
        # hover_name="primary_property_type_epa_calculated" if "primary_property_type_epa_calculated" in df.columns else None,
        hover_data=hover_cols,
        zoom=10,
        height=700,
        title=title,
        opacity=0.7,
        color_continuous_scale="Turbo",
    )
    fig.update_layout(mapbox_style="carto-positron", margin={"l": 0, "r": 0, "t": 60, "b": 0})
    status_html.value = ""
    fig.show()
    return fig

controls_top = widgets.HBox([dataset_dropdown])
controls_mid = widgets.HBox([color_dropdown])
# controls_filters = widgets.HBox([community_multi, property_multi])


interactive_out = widgets.interactive_output(
    make_interactive_map,
    {
        "dataset_name": dataset_dropdown,
        "color_col": color_dropdown,
        # "size_col": size_dropdown,
        # "selected_community": community_multi,
        "selected_property": property_multi
    },
)

display(controls_top, controls_mid, status_html, interactive_out)

HTML(value='')

Output()